In [ ]:
import tiktoken
import re
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn


In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

SMOKE_TEST_CONFIG = {
    "vocab_size": 50257,
    "context_length": 4,
    "emb_dim": 16,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.0,
    "qkv_bias": False,
}


In [ ]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str2int = vocab
        self.int2str = {i : s for s, i in vocab.items()}
    def encode(self, text):
        preprocessed = re.split(r"([,.:;?_!()'\"]|--|\s)", text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        return [self.str2int[token] if token in self.str2int else self.str2int["<|unk|>"]  
                for token in preprocessed]
    def decode(self, ids):
        text = " ".join([self.int2str[i] for i in ids])
        text = re.sub(r"\s+([,.:;?_!()'\"]|--)", r'\1', text)
        return text


模块名：dataset
输入：text
输出：(input_ids, output_ids)的tensor
内部状态：text-ids, input_ids, output_ids
核心步骤：先str2ids得到input，再根据滑窗取target
边界情况：文本长度小于数据集长度


In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]


In [ ]:
#data loader负责取dataset
def create_dataloader(
        text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader
text = "hello world,hello world, this is a small test text for the dataloader"


In [ ]:
class GPTInputEmbedding(nn.Module):
    def __init__(self, vocab, emb_dim, context_length):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab, emb_dim)
        self.position_embedding = nn.Embedding(context_length, emb_dim)

    def forward(self, token_ids):
        seq_len = token_ids.shape[1]
        positions = torch.arange(seq_len, device=token_ids.device)
        token_embeds = self.token_embedding(token_ids)
        pos_embeds = self.position_embedding(positions)
        input_embeds = token_embeds + pos_embeds
        return input_embeds


In [ ]:
#实现attention
class CausalAttention(nn.Module):
    def __init__(self, emb_dim, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_q=nn.Linear(emb_dim, d_out, bias=qkv_bias)
        self.W_k=nn.Linear(emb_dim, d_out, bias=qkv_bias)
        self.W_v=nn.Linear(emb_dim, d_out, bias=qkv_bias)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length,context_length, dtype=torch.bool),
            diagonal=1
        )
        )
        self.dropout=nn.Dropout(dropout)
    
    def forward(self, emb_vec, return_attn=False):
        b, seq_len, emb_dim = emb_vec.shape
        queries = self.W_q(emb_vec) #[B, T, D]
        keys = self.W_k(emb_vec)  
        values = self.W_v(emb_vec)
        d_k = keys.shape[-1]
        attention_scores = queries @ keys.transpose(1,2) / (d_k**0.5) #[B, T, T]
        masked_att = attention_scores.masked_fill(self.mask[:seq_len, :seq_len], -torch.inf)
        attention_weights = torch.softmax(masked_att, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vec = attention_weights @ values
        if return_attn:
            return context_vec, attention_weights
        return context_vec


In [ ]:
#简易版multihead实现
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, emb_dim, context_length, d_out, dropout, num_head, qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList(CausalAttention(
            emb_dim=emb_dim,
            d_out=d_out,
            context_length=context_length,
            dropout=dropout,
            qkv_bias=qkv_bias
        )
        for i in range(0, num_head)
        )
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


In [ ]:
'''multihead完全版实现
初始化参数：d_in, d_out, context_length, dropout, num_heads, qkv_bias
forward 输入：x，形状 [batch_size, num_tokens, d_in]
forward 输出：context_vec，形状 [batch_size, num_tokens, d_out]
输出：合并版的上下文向量
核心步骤：
0.初始化得到最大qkv矩阵
1.根据输出维度和多头数量得到每一个注意力头的输出维度c
1.a.分为多头
2.计算相应的注意力分数以及softmax化和dropout
3.通过view方法把注意力转换为多头
3.把单头得到的上下文向量通过torch.view结合起来，contigus()方法进行reshape
4.通过线性投影层，输出上下文向量
'''
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, context_length, dropout, d_out, num_heads,qkv_bias=False) -> None:
        super().__init__()
        self.W_q=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.d_out= d_out #保存一下下面forward使用
        self.num_heads=num_heads#保存一下下面forward使用
        self.dropout=nn.Dropout(dropout)
        self.out_proje=nn.Linear(d_out,d_out)
        self.register_buffer(
            'mask',
            torch.triu(
                torch.ones(context_length,context_length, dtype=torch.bool),#注意写上bool值才能让这个矩阵变成布尔矩阵 
                diagonal=1)
        )
        assert d_out % num_heads == 0, "d_out 必须被 num_heads 整除"
        self.head_dim=d_out // num_heads #view期待init,/返回float，所以要用//
    def forward(self, x):
        batch_size, seq_length, d_in = x.shape #[B,T,I]
        d_out=self.d_out#d_out = self.W_q.shape[-1]错误，linear模块没有shape查询
        num_heads=self.num_heads
        queries=self.W_q(x).view(batch_size, seq_length, num_heads, self.head_dim).transpose(1,2)#[B,T,N,H]->[B,N,T,H]
        keys=self.W_k(x).view(batch_size, seq_length, num_heads, self.head_dim).transpose(1,2)
        values=self.W_v(x).view(batch_size, seq_length, num_heads, self.head_dim).transpose(1,2)
        d_k = keys.shape[-1]
        attention_scores = queries @ keys.transpose(2,3) / d_k**0.5 #[B,N,T,T]
        masked_attn=torch.masked_fill(attention_scores, self.mask[:seq_length, :seq_length], -torch.inf)
        attn_weight = torch.softmax(masked_attn, -1)
        attn_weight = self.dropout(attn_weight)
        s_context_vec = attn_weight @ values#[B, N, T, H]
        context_vec = s_context_vec.transpose(1,2).contiguous().view(batch_size, seq_length, d_out)#这里可以优化，上面已经保存self.d_out了，不用再重新赋值
        return self.out_proje(context_vec)


In [ ]:
B = 2
T = 4
emb_dim = 16
d_out = 68
num_heads = 4
context_length = 4

x = torch.randn(B, T, emb_dim)

mha = MultiHeadAttention(
    d_in=emb_dim,
    d_out=d_out,
    context_length=context_length,
    dropout=0.0,
    num_heads=num_heads
)

out = mha(x)
print(out.shape)
print(out)


In [ ]:
'''归一化和GELU激活函数、前馈神经网络实现
输入：x [batch_size, seq_length, d_out]
输出：处理后的x，维度不变，但-1维均值为0，方差为1（为了让之后梯度下降更快收敛、训练更稳定）
核心目标：
1.归一化函数，用mean和var实现，先将每个 token 的最后一维归一化为均值约 0、方差约 1，
再通过可学习的 scale 和 bias 做缩放和平移。 #最好定为class，因为它有可训练参数
2.GELU激活函数，输入输出shape不变
3.FNN内交替调用神经层和GELU，中间隐藏层d_out增加
'''
class LayerNorm(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_in)) #[d]
        self.bias = nn.Parameter(torch.zeros(d_in))
        self.eps = 1e-5
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)#[b,s,1]
        var = x.var(dim=-1, keepdim=True, unbiased=False) #用总体var，unbiased为false:确保d小的时候var为1
        norm_x = (x-mean) / (var+self.eps)**0.5 #[b,s,d],可用torch.sqrt(...)替换        
        return norm_x*self.scale + self.bias #每个vec的d维度对应乘上self.scale d的相应索引值再加上self.bias的相应d索引值
class GELU(nn.Module): #也写成class,方便后面sequential使用
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi, device=x.device)) * (x + 0.044715 * x**3)
        ))
class FeedForward(nn.Module):
    def __init__(self, emb_dim, hidden_mult=4):
        super().__init__()
        self.fnn = nn.Sequential(
            nn.Linear(emb_dim, hidden_mult * emb_dim),
            GELU(),
            nn.Linear(hidden_mult*emb_dim, emb_dim)
        )
    def forward(self, x):
        return self.fnn(x) #[b,s,d]


In [ ]:
b = 2
s= 4
e = 3
x = torch.randn(b,s,e)
norm = LayerNorm(e)
fnn = FeedForward(e)
print(fnn(x).shape)
print(norm(x).shape)


In [ ]:
#实现transformer、残差连接。输入：emb_dim,drop,tensor x,输出修改后的x。两个归一化层，两次drop，Pre-LN
class Transformer(nn.Module):
    def __init__(self, emb_dim, dropout, num_heads, context_length, qkv_bias=False):
        super().__init__()
        self.mha = MultiHeadAttention(
            d_in=emb_dim,
            context_length=context_length,
            d_out=emb_dim,
            dropout=dropout,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
        )
        self.ffn=FeedForward(emb_dim)
        self.norm1=LayerNorm(emb_dim)
        self.norm2=LayerNorm(emb_dim)
        self.dropout=nn.Dropout(dropout)

    def forward(self, x):
        #attn
        shortcut = x
        x = self.norm1(x)
        x = self.mha(x)
        x = self.dropout(x)
        x += shortcut
        #ff
        shortcut = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = self.dropout(x)
        x += shortcut
        return x


In [ ]:
#GPT实现
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.input_embedding = GPTInputEmbedding(
            vocab=cfg["vocab_size"],
            emb_dim=cfg["emb_dim"],
            context_length=cfg["context_length"]
        )

        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[Transformer(
                emb_dim=cfg["emb_dim"],
                dropout=cfg["drop_rate"],
                num_heads=cfg["n_heads"],
                context_length=cfg["context_length"],
                qkv_bias=cfg["qkv_bias"],
            ) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False
        )

    def forward(self, in_idx):
        x = self.input_embedding(in_idx)   # [B, T, emb_dim]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)          # [B, T, vocab_size]
        return logits


In [ ]:
#用小配置把上面的模块串起来，确认 GPT 前向流程能跑通
model = GPTModel(SMOKE_TEST_CONFIG)
input_ids = torch.randint(
    low=0,
    high=SMOKE_TEST_CONFIG["vocab_size"],
    size=(2, SMOKE_TEST_CONFIG["context_length"]),
)
logits = model(input_ids)
print(logits.shape)
assert logits.shape == (2, SMOKE_TEST_CONFIG["context_length"], SMOKE_TEST_CONFIG["vocab_size"])
